In [1]:
import pandas as pd

# 1. Load the dataset into a Pandas DataFrame
df = pd.read_csv('owid-covid-data.csv')

# 2. Get Number of records (rows) and variables (columns)
print(f"Number of Records: {df.shape[0]}")
print(f"Number of Variables: {df.shape[1]}")

# 3. Get Data Types
df.info()

Number of Records: 350085
Number of Variables: 67
<class 'pandas.DataFrame'>
RangeIndex: 350085 entries, 0 to 350084
Data columns (total 67 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   iso_code                                    350085 non-null  str    
 1   continent                                   333420 non-null  str    
 2   location                                    350085 non-null  str    
 3   date                                        350085 non-null  str    
 4   total_cases                                 312088 non-null  float64
 5   new_cases                                   340457 non-null  float64
 6   new_cases_smoothed                          339198 non-null  float64
 7   total_deaths                                290501 non-null  float64
 8   new_deaths                                  340511 non-null  float64
 9   new_deaths_smoothed            

In [2]:
# Filter dataset for Asia (excluding non-country region entries if any exist)
asia_df = df[(df['continent'] == 'Asia') & (df['location'].notna())]

# Get unique country names
asia_countries = sorted(asia_df['location'].unique())

# Get total count
asia_country_count = len(asia_countries)

print(f"Total count of countries/territories in Asia: {asia_country_count}")
print("\nList of countries/territories in Asia:")
for country in asia_countries:
    print(f"- {country}")

Total count of countries/territories in Asia: 51

List of countries/territories in Asia:
- Afghanistan
- Armenia
- Azerbaijan
- Bahrain
- Bangladesh
- Bhutan
- Brunei
- Cambodia
- China
- Georgia
- Hong Kong
- India
- Indonesia
- Iran
- Iraq
- Israel
- Japan
- Jordan
- Kazakhstan
- Kuwait
- Kyrgyzstan
- Laos
- Lebanon
- Macao
- Malaysia
- Maldives
- Mongolia
- Myanmar
- Nepal
- North Korea
- Northern Cyprus
- Oman
- Pakistan
- Palestine
- Philippines
- Qatar
- Saudi Arabia
- Singapore
- South Korea
- Sri Lanka
- Syria
- Taiwan
- Tajikistan
- Thailand
- Timor
- Turkey
- Turkmenistan
- United Arab Emirates
- Uzbekistan
- Vietnam
- Yemen


In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# 1. Convert 'date' from string to datetime format (Fixing format)
df['date'] = pd.to_datetime(df['date'])

# 2. Filter the dataset to only include the continent of Asia
df_asia = df[df['continent'] == 'Asia'].copy()

# 3. Define our core metrics to audit
core_metrics = ['total_cases', 'total_deaths', 'total_vaccinations', 'icu_patients']

# 4. Calculate the percentage of MISSING data for each country
# (Formula: Missing rows / Total rows * 100)
missing_data_pct = df_asia.groupby('location')[core_metrics].apply(
    lambda x: (x.isnull().sum() / len(x)) * 100
).round(1)

# 5. Sort the list to show countries with the LEAST missing data at the top, and print the top 20
print("Percentage of Missing Data (%) per Country:")
print(missing_data_pct.sort_values(by=['icu_patients', 'total_vaccinations']).head(51))

Percentage of Missing Data (%) per Country:
                      total_cases  total_deaths  total_vaccinations  icu_patients
location                                                                         
Malaysia                      1.9           5.5                30.5           5.9
South Korea                   1.2           3.5                52.8          23.0
Israel                        3.0           5.6                38.9          29.3
Japan                         0.8           3.0                42.7          87.0
India                         2.3           5.4                29.1         100.0
Hong Kong                   100.0         100.0                32.2         100.0
Macao                       100.0         100.0                43.6         100.0
Kyrgyzstan                   39.1          39.1                46.9         100.0
China                         0.1           0.6                51.0         100.0
Singapore                     1.5          11.7       

In [4]:
# 1. Filter dataset to our 4 High-Fidelity countries
selected_countries = ['Malaysia', 'South Korea', 'Israel', 'Japan']
df_final = df_asia[df_asia['location'].isin(selected_countries)].copy()

# 2. Sort chronologically per country (Crucial before any time-series math)
df_final = df_final.sort_values(by=['location', 'date']).reset_index(drop=True)

# 3. Check and remove duplicates (Rubric 3b)
duplicates = df_final.duplicated().sum()
df_final = df_final.drop_duplicates()

# 4. Separate columns by their statistical behavior
cumulative_cols = ['total_cases', 'total_deaths', 'total_vaccinations']
snapshot_cols = ['icu_patients', 'hosp_patients', 'new_cases', 'new_deaths']

# 5. Apply Forward Fill to cumulative totals (Grouped by country so data doesn't bleed!)
df_final[cumulative_cols] = df_final.groupby('location')[cumulative_cols].ffill()

# 6. Apply Linear Interpolation to fluctuating snapshots (bridging gaps up to 14 days)
df_final[snapshot_cols] = df_final.groupby('location')[snapshot_cols].transform(
    lambda x: x.interpolate(method='linear', limit=14)
)

# 7. Fill remaining NaNs (e.g., early 2020 before vaccines/cases existed) with 0
all_core_cols = cumulative_cols + snapshot_cols
df_final[all_core_cols] = df_final[all_core_cols].fillna(0)

# 8. Final Audit
print(f"Selected Countries: {df_final['location'].unique().tolist()}")
print(f"Total Rows: {df_final.shape[0]}")
print(f"Duplicates Removed: {duplicates}")
print("\nMissing values remaining in core metrics:")
print(df_final[all_core_cols].isnull().sum())

Selected Countries: ['Israel', 'Japan', 'Malaysia', 'South Korea']
Total Rows: 5543
Duplicates Removed: 0

Missing values remaining in core metrics:
total_cases           0
total_deaths          0
total_vaccinations    0
icu_patients          0
hosp_patients         0
new_cases             0
new_deaths            0
dtype: int64


In [5]:
# 1. Generate Descriptive Statistics for our core metrics (Rubric 3b & 3c)
stats_summary = df_final[all_core_cols].describe().round(2)

# 2. Check specifically if any rows have negative (< 0) values
negative_counts = (df_final[all_core_cols] < 0).sum()

print("=== COUNT OF INVALID NEGATIVE VALUES ===")
print(negative_counts)
print("\n=== DESCRIPTIVE STATISTICS (MIN, MAX, MEAN, QUARTILES) ===")
print(stats_summary.loc[['min', 'mean', '50%', '75%', 'max']])

=== COUNT OF INVALID NEGATIVE VALUES ===
total_cases           0
total_deaths          0
total_vaccinations    0
icu_patients          0
hosp_patients         0
new_cases             0
new_deaths            0
dtype: int64

=== DESCRIPTIVE STATISTICS (MIN, MAX, MEAN, QUARTILES) ===
      total_cases  total_deaths  total_vaccinations  icu_patients  hosp_patients  new_cases  new_deaths
min          0.00          0.00        0.000000e+00          0.00           0.00        0.0        0.00
mean   6741715.89      17413.78        7.434336e+07        316.18        3836.14    14134.4       28.97
50%    1711935.00      10896.00        1.864321e+07        121.00         733.00     1252.0        6.00
75%    5052337.00      31884.00        1.196004e+08        377.00        3790.21     6710.5       26.00
max   34571873.00      74694.00        3.837477e+08       3034.00       44458.00   621328.0      592.00
